# Unit 3 solutions: predict where the car will be

These reference solutions build the kinematic car's model from its parameters, predict each answer first where the lesson gives a law (the circle, the quadratic, the half-step lag, the error orders), and then measure it.
Every race is deterministic or seeded, so every number below is reproducible, and executable asserts check each answer against an independent derivation.

In [ ]:
import math
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from racekit.cars import KinematicCar
from racekit.runner import CONTROL_DT, run_race
from racekit.tracks import load_track

STARTER = Path("../../projects/racekit/agents/unit-01.py")

# racekit's kinematic car (a model, not the truth)
WHEELBASE = 2.875  # m
DELTA_MAX = math.radians(70.0)  # wheel angle at |steer| = 1; delta = -steer * DELTA_MAX
A_THROTTLE = 6.0  # m/s^2 at full throttle
A_BRAKE = 8.0  # m/s^2 at full brake, against the motion
C_DRAG = 4.0e-4  # 1/m: drag deceleration C_DRAG * v * |v|

## Exercise 1 - The circle, and the steer a corner needs

A steer of −0.1 gives δ = −(−0.1)·δ_max = +0.1·70° = 7°, a **positive** wheel angle, so the car turns **left**.
R = L / tan δ = 2.875 / tan 7° ≈ 23.415 m (the lesson's radius for |steer| = 0.1), ψ̇ = v / R ≈ +0.641 rad/s, and one circle takes 2πR / v ≈ 9.81 s.

For the corners, the Menger curvature is 2·(signed twice-area) / (product of the three sides), and the steer is the inverse of κ = tan(−steer·δ_max) / L: steer = −arctan(L κ) / δ_max.
A left corner (κ > 0) needs a negative steer; a right corner a positive one.

In [ ]:
steer, speed = -0.1, 15.0
delta = -steer * DELTA_MAX
kappa_circle = math.tan(delta) / WHEELBASE  # signed, left positive
radius = 1.0 / abs(kappa_circle)
yaw_rate = speed * kappa_circle
turns_left = kappa_circle > 0.0
lap_time = 2.0 * math.pi * radius / speed


def signed_menger(xs, ys):
    # The signed curvature at every waypoint from (i - 1, i, i + 1), cyclic, left positive.
    n = len(xs)
    out = np.zeros(n)
    for i in range(n):
        a, c = (i - 1) % n, (i + 1) % n
        twice_area = (xs[i] - xs[a]) * (ys[c] - ys[a]) - (ys[i] - ys[a]) * (xs[c] - xs[a])
        sides = (math.hypot(xs[i] - xs[a], ys[i] - ys[a]) * math.hypot(xs[c] - xs[i], ys[c] - ys[i])
                 * math.hypot(xs[c] - xs[a], ys[c] - ys[a]))
        out[i] = 2.0 * twice_area / sides
    return out


steer_needed = {}
tightest_kappa = {}
for name in ("oval", "test-circuit", "monza-like"):
    track = load_track(name)
    kappa = signed_menger(track.x, track.y)
    tightest = int(np.argmax(np.abs(kappa)))
    tightest_kappa[name] = float(kappa[tightest])
    steer_needed[name] = -math.atan2(WHEELBASE * kappa[tightest], 1.0) / DELTA_MAX

print(f"delta = {delta:.6f} rad ({math.degrees(delta):.1f} deg), R = {radius:.4f} m, yaw rate = {yaw_rate:+.5f} rad/s")
print(f"turns left: {turns_left}; one circle at 15 m/s takes {lap_time:.4f} s")
for name, value in steer_needed.items():
    print(f"{name:>12}: tightest kappa {tightest_kappa[name]:+.5f} 1/m (R = {1 / abs(tightest_kappa[name]):.2f} m), steer {value:+.5f}")

assert abs(math.degrees(delta) - 7.0) < 1e-12 and turns_left and yaw_rate > 0.0
assert abs(radius - 23.415) < 5e-4 and abs(yaw_rate - speed / radius) < 1e-12
assert abs(lap_time - 2.0 * math.pi / yaw_rate) < 1e-12
assert abs(abs(steer_needed["test-circuit"]) - 0.1296) < 5e-4  # the lesson's number
for name, value in steer_needed.items():
    # The sign rule: a left corner (kappa > 0) needs a negative steer, and the steer reproduces kappa.
    assert (value < 0.0) == (tightest_kappa[name] > 0.0)
    assert abs(math.tan(-value * DELTA_MAX) / WHEELBASE - tightest_kappa[name]) < 1e-12

## Exercise 2 - The longitudinal map and the terminal speed

At full throttle v̇ = 0 when a_T = c·v², so v = √(a_T / c) = √15000 ≈ 122.47 m/s.
Below 20 m/s the starter's brake is 0 and its throttle is 0.05·(20 − v), so the balance is c·v² + 0.3·v − 6 = 0, with positive root ≈ 19.4933 m/s.
The simulation must approach that root from below and sit on it.

Braking from 20 m/s for 2 s with no drag is 20·2 − ½·8·2² = 24 m, and the speed is still 4 m/s, so the brake never reaches the stop.
Drag only adds deceleration, so the real distance must be a little **less** than 24 m.

In [ ]:
def acceleration(v, throttle, brake):
    accel = A_THROTTLE * throttle - C_DRAG * v * abs(v)
    if v > 0.0:
        accel -= A_BRAKE * brake
    elif v < 0.0:
        accel += A_BRAKE * brake
    return accel


v_terminal = math.sqrt(A_THROTTLE / C_DRAG)

# c v^2 + 0.3 v - 6 = 0
qa, qb, qc = C_DRAG, A_THROTTLE * 0.05, -A_THROTTLE * 0.05 * 20.0
v_equilibrium = (-qb + math.sqrt(qb * qb - 4.0 * qa * qc)) / (2.0 * qa)


def speed_rk4(v, throttle, brake, h):
    # One RK4 step of v' = acceleration(v, throttle, brake), the action held.
    k1 = acceleration(v, throttle, brake)
    k2 = acceleration(v + 0.5 * h * k1, throttle, brake)
    k3 = acceleration(v + 0.5 * h * k2, throttle, brake)
    k4 = acceleration(v + h * k3, throttle, brake)
    return v + h / 6.0 * (k1 + 2.0 * k2 + 2.0 * k3 + k4)


v = 0.0
for _ in range(1200):
    throttle = float(np.clip(0.05 * (20.0 - v), 0.0, 1.0))
    brake = float(np.clip(-0.05 * (20.0 - v), 0.0, 1.0))
    v = speed_rk4(v, throttle, brake, 0.05)
v_after_60s = v

# Full braking along a straight line: integrate (x, v) together with RK4.
def line_derivatives(xv):
    return np.array([xv[1], acceleration(xv[1], 0.0, 1.0)])


xv = np.array([0.0, 20.0])
h = 0.05
for _ in range(40):
    k1 = line_derivatives(xv)
    k2 = line_derivatives(xv + 0.5 * h * k1)
    k3 = line_derivatives(xv + 0.5 * h * k2)
    k4 = line_derivatives(xv + h * k3)
    xv = xv + h / 6.0 * (k1 + 2.0 * k2 + 2.0 * k3 + k4)
brake_distance = float(xv[0])
brake_distance_no_drag = 20.0 * 2.0 - 0.5 * A_BRAKE * 2.0 ** 2

why_below_target = (
    "A proportional controller only pushes when there is a speed error, and holding any speed against drag "
    "needs a push, so the starter settles where 6 x 0.05 x (20 - v) = 4e-4 v^2, at about 19.49 m/s: "
    "the 'a little below 20 m/s' steady speed Unit 1 measured in its Exercise 6."
)

print(f"terminal speed {v_terminal:.4f} m/s | equilibrium root {v_equilibrium:.6f} m/s | simulated after 60 s {v_after_60s:.6f} m/s")
print(f"braking 2 s from 20 m/s: {brake_distance:.5f} m with drag (speed now {xv[1]:.4f} m/s), {brake_distance_no_drag:.1f} m without")
assert abs(v_terminal - 122.47) < 0.005
assert abs(acceleration(v_equilibrium, 0.05 * (20.0 - v_equilibrium), 0.0)) < 1e-12
assert abs(v_equilibrium - 19.49) < 0.005 and abs(v_after_60s - v_equilibrium) < 1e-5
assert brake_distance_no_drag == 24.0 and 23.0 < brake_distance < brake_distance_no_drag
assert 0.0 < xv[1] < 4.0  # still rolling, slower than the no-drag 4 m/s
assert acceleration(0.0, 0.0, 1.0) == 0.0 and acceleration(-5.0, 0.0, 1.0) > 0.0

## Exercise 3 - Euler's half-step lag

**Prediction (written before measuring).**
With v and the steer held, every Euler step is a side of length h·v of a regular polygon tangent to the true circle at the start, so the whole circle is shifted forward by about h·v/2.
That predicts 0.05·10/2 = 0.25 m, 0.025·10/2 = 0.125 m and 0.05·20/2 = 0.5 m.

The measurement agrees to a few millimetres: the polygon's own circumradius exceeds R by about R·(hv/2R)²/6, and a shifted circle's distance from the true centre swings between R − shift and R + shift once per lap.

In [ ]:
def derivatives(state, action):
    x, y, psi, v = state
    throttle, steer, brake = action
    kappa = math.tan(-steer * DELTA_MAX) / WHEELBASE
    return np.array([v * math.cos(psi), v * math.sin(psi), v * kappa, acceleration(v, throttle, brake)])


def euler_step(state, action, h):
    return state + h * derivatives(state, action)


def circle_shift(step, steer, h, v, steps):
    # The maximum over all steps of |distance from the true centre (0, -R) - R| on a held circle.
    radius = WHEELBASE / math.tan(steer * DELTA_MAX)
    action = (C_DRAG * v * v / A_THROTTLE, steer, 0.0)
    state = np.array([0.0, 0.0, 0.0, v])
    worst = 0.0
    for _ in range(steps):
        state = step(state, action, h)
        worst = max(worst, abs(math.hypot(state[0], state[1] + radius) - radius))
    return worst


CASES = [(0.05, 10.0), (0.025, 10.0), (0.05, 20.0)]
predicted_shifts = [0.05 * 10.0 / 2.0, 0.025 * 10.0 / 2.0, 0.05 * 20.0 / 2.0]
measured_shifts = [circle_shift(euler_step, 0.1, h, v, int(round(100 / h))) for h, v in CASES]
scaling = (
    "The shift is about h v / 2, proportional to h and to v (halving h halves it, doubling v doubles it), "
    "because Euler's held heading lags the average heading over a step by half a step; with v and steer "
    "held every step is the same side of a regular polygon, so the error is a fixed shift of the whole "
    "circle that repeats every lap instead of accumulating."
)

for (h, v), predicted, measured in zip(CASES, predicted_shifts, measured_shifts):
    print(f"h = {h:5}, v = {v:4}: predicted {predicted:.4f} m, measured {measured:.6f} m")
for predicted, measured in zip(predicted_shifts, measured_shifts):
    assert abs(measured - predicted) < 0.02 * predicted
assert abs(measured_shifts[0] / measured_shifts[1] - 2.0) < 0.05
assert abs(measured_shifts[2] / measured_shifts[0] - 2.0) < 0.05
# The action really holds the speed: the drag-balancing throttle makes v' = 0.
assert abs(derivatives(np.array([0.0, 0.0, 0.0, 10.0]), (C_DRAG * 100.0 / A_THROTTLE, 0.1, 0.0))[3]) < 1e-15

## Exercise 4 - Measure the orders on a new manoeuvre

The prediction: Euler's error ratios near 2 (order 1) and RK4's near 16 (order 4), drifting at the largest h.
Full throttle from rest is a harder case than the lesson's: v changes from 0 to about 17 m/s while the turn tightens in yaw rate, so the higher-order terms matter more at h = 0.2.

For `ticks_for_euler`: a first-order error is C·h, so matching RK4's error at h = 0.025 needs h smaller by the factor e_Euler(0.025) / e_RK4(0.025), which is the factor of extra steps.

In [ ]:
def rk4_step(state, action, h):
    k1 = derivatives(state, action)
    k2 = derivatives(state + 0.5 * h * k1, action)
    k3 = derivatives(state + 0.5 * h * k2, action)
    k4 = derivatives(state + h * k3, action)
    return state + h / 6.0 * (k1 + 2.0 * k2 + 2.0 * k3 + k4)


def final_state(step, h, horizon, action, start):
    state = np.array(start)
    for _ in range(int(round(horizon / h))):
        state = step(state, action, h)
    return state


T = 3.0
LAUNCH = (1.0, -0.2, 0.0)
ORIGIN = (0.0, 0.0, 0.0, 0.0)
STEPS = np.array([0.2, 0.1, 0.05, 0.025])
reference = final_state(rk4_step, 1e-3, T, LAUNCH, ORIGIN)


def position_errors(step):
    return np.array([math.hypot(*(final_state(step, h, T, LAUNCH, ORIGIN)[:2] - reference[:2])) for h in STEPS])


euler_errors = position_errors(euler_step)
rk4_errors = position_errors(rk4_step)
euler_orders = np.log2(euler_errors[:-1] / euler_errors[1:])
rk4_orders = np.log2(rk4_errors[:-1] / rk4_errors[1:])
ticks_for_euler = float(euler_errors[-1] / rk4_errors[-1])

fig, ax = plt.subplots(figsize=(6, 4.5))
ax.loglog(STEPS, euler_errors, "o-", label="Euler")
ax.loglog(STEPS, rk4_errors, "s-", label="RK4")
ax.set_xlabel("step h (s)")
ax.set_ylabel("position error after 3 s (m)")
ax.set_title("Launch from rest at steer -0.2: error vs step size")
ax.grid(True, which="both", alpha=0.3)
ax.legend()
plt.show()

print("reference final state:", np.round(reference, 6))
print("Euler errors", euler_errors, "orders", np.round(euler_orders, 4))
print("RK4 errors  ", rk4_errors, "orders", np.round(rk4_orders, 4))
print(f"Euler needs {ticks_for_euler:.4g} times the steps of h = 0.025 to match RK4 there")
assert reference[3] > 15.0 and reference[2] > 0.0  # a left turn, accelerating
assert np.max(np.abs(euler_orders - 1.0)) < 0.15 and np.max(np.abs(rk4_orders - 4.0)) < 0.5
assert ticks_for_euler > 1e4

## Exercise 5 - Replay the telemetry, with and without noise

Prediction for `clean`: racekit's car **is** this model, integrated with ten RK4 substeps per tick, so a one-step RK4 replay is off only by RK4's own error at h = 0.05: nanometres.
Euler is first order, so one step per tick should be decimetres off in the worst corner window, and ten substeps should cut that about tenfold.

On `noisy` the integrator is not the problem.

In [ ]:
clean = run_race(STARTER, "test-circuit", car=KinematicCar(jitter=0.0))
noisy = run_race(STARTER, "test-circuit", car=KinematicCar(jitter=0.05), seed=0)


def wrap_angle(angle):
    return (angle + math.pi) % (2.0 * math.pi) - math.pi


def replay(telemetry, step, substeps=1):
    # 1 s predictions with the recorded actions: arrays of the position and wrapped heading errors.
    starts = np.arange(100, len(telemetry) - 20, 50)
    h = CONTROL_DT / substeps
    position, heading = np.zeros(len(starts)), np.zeros(len(starts))
    for m, row in enumerate(starts):
        state = np.array([telemetry["x"][row], telemetry["y"][row], telemetry["yaw"][row], telemetry["v"][row]])
        for i in range(row + 1, row + 21):
            action = (telemetry["throttle"][i], telemetry["steer"][i], telemetry["brake"][i])
            for _ in range(substeps):
                state = step(state, action, h)
        end = row + 20
        position[m] = math.hypot(state[0] - telemetry["x"][end], state[1] - telemetry["y"][end])
        heading[m] = wrap_angle(state[2] - telemetry["yaw"][end])
    return position, heading


rk4_clean, rk4_clean_heading = replay(clean.telemetry, rk4_step)
euler_clean, _ = replay(clean.telemetry, euler_step)
euler10_clean, _ = replay(clean.telemetry, euler_step, substeps=10)
rk4_noisy, _ = replay(noisy.telemetry, rk4_step)

rk4_max_clean = float(np.max(rk4_clean))
euler_max_clean = float(np.max(euler_clean))
euler10_max_clean = float(np.max(euler10_clean))
max_yaw_error_clean = float(np.max(np.abs(rk4_clean_heading)))
rk4_max_noisy = float(np.max(rk4_noisy))
rk4_median_noisy = float(np.median(rk4_noisy))
why_noisy = (
    "The telemetry records the action the agent asked for, before the jitter, but the car drove the "
    "jittered throttle and steer, so the replay integrates the wrong actions: the error is in the "
    "inputs, not the integrator."
)

print(clean, "|", noisy)
print(f"{len(rk4_clean)} windows on each race")
print(f"clean: RK4 max {rk4_max_clean:.3e} m, Euler max {euler_max_clean:.5f} m, Euler x10 max {euler10_max_clean:.6f} m, "
      f"RK4 max |heading error| {max_yaw_error_clean:.3e} rad")
print(f"noisy: RK4 max {rk4_max_noisy:.4f} m, median {rk4_median_noisy:.4f} m")
assert clean.finished and clean.collisions == 0 and np.sum(clean.telemetry["held"]) == 0
assert noisy.finished and noisy.collisions == 0 and np.sum(noisy.telemetry["held"]) == 0
assert rk4_max_clean < 1e-6 and max_yaw_error_clean < 1e-8
assert 9.0 < euler_max_clean / euler10_max_clean < 11.0 and euler10_max_clean > rk4_max_clean
assert rk4_max_noisy > 1e5 * rk4_max_clean and rk4_median_noisy > 0.01

## Exercise 6 - Where will the car be relative to the track?

Holding the current action ignores the steering the starter will add or remove within the second.
On a straight that is almost free; entering or leaving a corner the predicted path has the wrong curvature, so the miss should be mostly **sideways** (the turn the car did not make), and largest at corner entries.
Along the heading, the speed hardly changes (the starter sits near 19.49 m/s), so the forward component should be small.

The car-frame rotation is Unit 2's `to_car_frame` applied to the error vector, and `cross_track_error` is Unit 2's windowed nearest segment plus the signed offset, written to the pinned rules.

In [ ]:
def cross_track_error(px, py, xs, ys, hint, reach=8):
    n = len(xs)
    best_d2, best = math.inf, hint % n
    for k in range(hint - reach, hint + reach + 1):
        i = k % n
        j = (i + 1) % n
        dx, dy = xs[j] - xs[i], ys[j] - ys[i]
        t = ((px - xs[i]) * dx + (py - ys[i]) * dy) / (dx * dx + dy * dy)
        t = max(0.0, min(t, 1.0))
        fx, fy = px - (xs[i] + t * dx), py - (ys[i] + t * dy)
        if fx * fx + fy * fy < best_d2:
            best_d2, best = fx * fx + fy * fy, i
    j = (best + 1) % n
    dx, dy = xs[j] - xs[best], ys[j] - ys[best]
    ex, ey = px - xs[best], py - ys[best]
    return best, (dx * ey - dy * ex) / math.sqrt(dx * dx + dy * dy)


track = load_track("test-circuit")
xs, ys = track.x, track.y
telemetry = clean.telemetry
starts = np.arange(100, len(telemetry) - 20, 50)
along, lateral, predicted_offsets, offset_gaps = [], [], [], []
recorded_check = []
for row in starts:
    state = np.array([telemetry["x"][row], telemetry["y"][row], telemetry["yaw"][row], telemetry["v"][row]])
    action = (telemetry["throttle"][row + 1], telemetry["steer"][row + 1], telemetry["brake"][row + 1])
    for _ in range(20):
        state = rk4_step(state, action, CONTROL_DT)
    end = row + 20
    rx, ry, ryaw = telemetry["x"][end], telemetry["y"][end], telemetry["yaw"][end]
    ex, ey = state[0] - rx, state[1] - ry
    along += [math.cos(ryaw) * ex + math.sin(ryaw) * ey]
    lateral += [-math.sin(ryaw) * ex + math.cos(ryaw) * ey]
    hint = int(np.argmin(np.hypot(xs - rx, ys - ry)))
    _, offset = cross_track_error(state[0], state[1], xs, ys, hint)
    predicted_offsets += [offset]
    offset_gaps += [offset - telemetry["offset"][end]]
    recorded_check += [cross_track_error(rx, ry, xs, ys, hint)[1] - telemetry["offset"][end]]
along, lateral = np.array(along), np.array(lateral)
predicted_offsets, offset_gaps = np.array(predicted_offsets), np.array(offset_gaps)

along_rms = float(np.sqrt(np.mean(along ** 2)))
lateral_rms = float(np.sqrt(np.mean(lateral ** 2)))
max_predicted_offset = float(np.max(np.abs(predicted_offsets)))
max_offset_gap = float(np.max(np.abs(offset_gaps)))
dominant = "along" if along_rms > lateral_rms else "lateral"
narrowest = float(min(track.half_width))
what_it_means = (
    f"Near a corner the held-action prediction puts the car up to {max_offset_gap:.1f} m across the track from "
    f"where it really ends up (up to {max_predicted_offset:.1f} m off the centreline, against a narrowest "
    f"half-width of {narrowest:.1f} m), so a controller that trusted it would conclude the car is running "
    f"toward the edge and brake or over-correct, when the steering the starter is about to add takes it round."
)

print(f"{len(starts)} windows | along RMS {along_rms:.4f} m, lateral RMS {lateral_rms:.4f} m -> {dominant}")
print(f"max |predicted offset| {max_predicted_offset:.4f} m, max |offset gap| {max_offset_gap:.4f} m, narrowest half-width {narrowest:.2f} m")
print(what_it_means)
# The function reproduces racekit's recorded offset when given the recorded position.
assert np.max(np.abs(recorded_check)) < 1e-6
assert len(starts) == len(rk4_clean)
assert max_offset_gap <= max_predicted_offset + np.max(np.abs(telemetry["offset"]))
assert dominant in ("along", "lateral")

## Exercise 7 - Challenge: how small must Euler's step be?

If the error were exactly C·h, then e(n) = e(20)·20 / n, and e(n) < 0.01 m first when n > 20·e(20) / 0.01.
The true curve is not exactly first order: the second-order terms make the real error a little different, so the measured smallest n can disagree with the prediction by a few steps.
Here the prediction is 820 and the measurement 819: the law is almost exact, and Euler at 1 cm still costs about 10 times RK4's work at the tick for an error about 450 000 times larger.

In [ ]:
SPIN = (0.0, 0.1, 0.0)
FAST = (0.0, 0.0, 0.0, 20.0)
spin_reference = final_state(rk4_step, 1e-4, 1.0, SPIN, FAST)


def spin_error(step, n):
    state = np.array(FAST)
    for _ in range(n):
        state = step(state, SPIN, 1.0 / n)
    return math.hypot(state[0] - spin_reference[0], state[1] - spin_reference[1])


error_at_tick = spin_error(euler_step, 20)
x = 20.0 * error_at_tick / 0.01
n_predicted = int(x) + 1

# Bisection: lo always fails (error >= 1 cm), hi always passes.
lo, hi = 20, 40
while spin_error(euler_step, hi) >= 0.01:
    lo, hi = hi, 2 * hi
while hi - lo > 1:
    mid = (lo + hi) // 2
    if spin_error(euler_step, mid) < 0.01:
        hi = mid
    else:
        lo = mid
n_actual = hi
h_star = 1.0 / n_actual
rk4_error_at_tick = spin_error(rk4_step, 20)
cost_verdict = (
    f"Euler needs {n_actual} evaluations of f for 1 cm, while RK4 at the race tick costs 4 x 20 = 80 "
    f"evaluations for an error of {rk4_error_at_tick:.1e} m, about {n_actual / 80:.0f} times cheaper and "
    f"far more accurate; the first-order prediction ({n_predicted}) was {n_predicted - n_actual} step too many, "
    f"because at h = 0.05 the error is not exactly proportional to h: the h^2 term still adds a little "
    f"to error_at_tick, so scaling it down linearly slightly overestimates the error at small h."
)

print(f"Euler error at n = 20: {error_at_tick:.6f} m | predicted n {n_predicted} (x = {x:.3f}) | actual n {n_actual}, h* = {h_star:.3e} s")
print(f"RK4 error at n = 20: {rk4_error_at_tick:.3e} m")
print(cost_verdict)
assert spin_error(euler_step, n_actual) < 0.01 <= spin_error(euler_step, n_actual - 1)
assert x != int(x) and n_predicted - 1 < x < n_predicted
assert rk4_error_at_tick < 1e-4
assert abs(n_actual - n_predicted) <= 5  # the first-order law is nearly right

## Exercise 8 - Challenge: semi-implicit Euler, and its order

Updating v and ψ first and then moving with the new values is still a one-evaluation method whose local error is O(h²), so its global order should still be **1**.
What changes is the constant: on a circle, moving along the **end** heading instead of the start heading turns the half-step lag into a half-step lead, and the two errors are the same size with the opposite sign: expect a shift of about h·v/2 = 0.25 m again.
On the lesson's accelerating case the v-update also helps, because the position step uses the new speed, so the constant can differ from Euler's.

In [ ]:
def semi_implicit_step(state, action, h):
    x, y, psi, v = state
    rates = derivatives(state, action)
    v_new = v + h * rates[3]
    psi_new = psi + h * rates[2]
    return np.array([x + h * v_new * math.cos(psi_new), y + h * v_new * math.sin(psi_new), psi_new, v_new])


LESSON_CASE = (0.6, 0.15, 0.0)
LESSON_START = (0.0, 0.0, 0.0, 5.0)
LESSON_STEPS = np.array([0.4, 0.2, 0.1, 0.05])
lesson_reference = final_state(rk4_step, 1e-4, 4.0, LESSON_CASE, LESSON_START)


def lesson_errors(step):
    return np.array([math.hypot(*(final_state(step, h, 4.0, LESSON_CASE, LESSON_START)[:2] - lesson_reference[:2]))
                     for h in LESSON_STEPS])


semi_errors = lesson_errors(semi_implicit_step)
semi_orders = np.log2(semi_errors[:-1] / semi_errors[1:])
euler_errors_lesson = lesson_errors(euler_step)
constant_ratio = float(euler_errors_lesson[-1] / semi_errors[-1])
semi_circle_shift = circle_shift(semi_implicit_step, 0.1, 0.05, 10.0, 2000)
order_verdict = (
    f"Semi-implicit Euler is still a first-order method (observed orders {np.round(semi_orders, 2)}), "
    f"but here its error constant is {constant_ratio:.2f} times smaller than Euler's for the same cost; "
    f"on the circle it is {semi_circle_shift:.3f} m off, the same half-step error as Euler's, only a lead "
    f"instead of a lag, so it is not better there."
)

print("semi-implicit errors", semi_errors, "orders", np.round(semi_orders, 4))
print("Euler errors        ", euler_errors_lesson, "| ratio at h = 0.05:", round(constant_ratio, 4))
print(f"circle: max |dist - R| = {semi_circle_shift:.6f} m")
print(order_verdict)
assert abs(euler_errors_lesson[-1] - 1.05) < 0.01  # the lesson's number
assert np.max(np.abs(semi_orders - 1.0)) < 0.3
assert abs(semi_circle_shift - 0.25) < 0.01